# Get NISAR GCOV Backscatter Time Series per ISMN Station

<br>

For each ISMN soil moisture station in `ismn_data/spain/stations.geojson`, **processed
independently of the others**, this notebook builds a per-station time series of NISAR
L-band (frequency A) HH/HV backscatter over a 3x3 pixel window centered on the pixel the
station falls into, reporting the window's **mean and standard deviation in dB**
(`NISAR_L_HH_db_mean3x3`/`NISAR_L_HH_db_std3x3`, and the HV equivalents). Each
acquisition's **local (terrain-corrected) incidence angle** is also recorded (as
`NISAR_local_incidence_angle_deg`) — see the note below for why the ellipsoidal
incidence angle is not used directly as a feature.

One CSV is written per station: `{station_name}_NISAR_raw.csv`, with columns:

`datetime_utc`, `NISAR_L_HH_db_mean3x3`, `NISAR_L_HH_db_std3x3`,
`NISAR_L_HV_db_mean3x3`, `NISAR_L_HV_db_std3x3`, `NISAR_local_incidence_angle_deg`,
`NISAR_orb_dir`, `NISAR_orb_number`

It uses **Option 2 (stream over HTTPS)** from
[`search_NISAR_GCOV_earthaccess.ipynb`](../notebooks/search_NISAR_GCOV_earthaccess.ipynb):
each GCOV granule is opened remotely with `fsspec` + `h5py` — no full-granule
downloads (each granule is ~1.7 GB).

:::{note} Changes from the previous version of this notebook
- **Simplified to a single 3x3-window mean/std, calibration removed.** The previous
  version of this notebook wrote raw single-pixel values, 3x3/5x5 window means, and
  all of those again calibrated to a 40° reference angle via a cosine power law — nine
  backscatter-derived columns per polarization in total. This is now reduced to just
  `NISAR_L_HH_db_mean3x3`/`NISAR_L_HH_db_std3x3` (and the HV equivalents): the mean is
  still computed by averaging the window's linear power values then converting to dB
  (dB values can't be arithmetically averaged directly), and the standard deviation is
  computed directly on the per-pixel dB values (sample std, ddof=1) as a texture/spread
  descriptor — there's no clean power-domain equivalent for a spread statistic the way
  there is for a mean. Single-pixel values, the 5x5 window, and the calibrated columns
  (previously validated only against this cookbook's Austria dataset — see the note
  below on that check's premise) are no longer computed at all. **This change is in
  the shared module** (`util/nisar_station_timeseries.py`), so it also applies to this
  cookbook's other AOI notebooks (Austria, USA high/low biomass).
- **Extraction pipeline moved to a shared module.** The DEM-terrain, dedup, track
  filter, same-pass merge, and per-granule extraction logic below — previously
  copy-pasted separately into each of this cookbook's `02_get_NISAR_*_data.ipynb`
  notebooks — now lives in
  [`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py),
  imported here. Fixes and behavior changes (like the ones in this note) now apply to
  every AOI notebook from one place instead of needing to be manually repeated
  per notebook. Each section below still documents what its step does
  conceptually; the code cell under it just imports or calls the shared
  implementation rather than redefining it.
- **Local (DEM-corrected) incidence angle instead of the ellipsoidal one.** GCOV's
  own `incidenceAngle`/`elevationAngle` cubes are both defined relative to the smooth
  WGS84 **ellipsoid normal** (at the target and at the sensor respectively) — neither
  accounts for actual terrain tilt. Exploratory analysis (`NISAR_orbit_exploration.ipynb`
  in the main modeling repo) found a large per-orbit-track effect on raw backscatter
  that the ellipsoidal incidence angle didn't explain, and adding the ellipsoidal
  elevation angle alongside it didn't explain it either. The next candidate is
  terrain-facing geometry: a slope facing toward the radar is illuminated more
  directly (smaller local incidence angle, typically brighter) than the same slope
  facing away, at *identical* ellipsoidal incidence angle — an effect neither
  ellipsoidal angle can capture. This notebook computes the true local incidence angle
  by combining GCOV's per-acquisition line-of-sight vector (`losUnitVectorX/Y`) with
  per-station terrain slope/aspect from the **Copernicus GLO-30 DEM** (see the new
  section below), and writes out only `NISAR_local_incidence_angle_deg` —
  `NISAR_incidence_angle_deg` (the old ellipsoidal column) is no longer written.
- **Independent per-station processing.** Each station now runs its own CMR search
  (via `point=`, so CMR itself does the spatial filtering — no more shared
  bounding-box search followed by a separate polygon-matching step) and is fully
  extracted and saved before moving to the next station. The tradeoff: granules
  covering multiple nearby stations are now opened once per station rather than once
  total, but each open only streams a few pixels, so this is cheap.
- **Product version deduplication**, adapted from
  [`NISAR_GCOV_Time_Series_Example.ipynb`](../notebooks/NISAR_GCOV_Time_Series_Example.ipynb):
  if a granule is reprocessed, CMR can return more than one CRID
  (composite-release-identifier) version of the same acquisition. We now keep only the
  highest CRID per unique acquisition, so reprocessing doesn't create duplicate rows.
- **Defensive dB conversion**, also adapted from that notebook: `log10` is only
  applied to strictly positive values (`ts.where(ts > 0)` there; an equivalent guard
  here), avoiding `-inf`/warnings on zero-valued pixels.
- **Merging same-overpass adjacent frames.** A station near a frame boundary can be
  covered by two granules from the *same physical pass*, acquired only seconds apart
  (same `orb_number`). These are now merged into a single row: `..._mean3x3` is
  averaged **in linear power, not dB** — dB values can't be arithmetically averaged
  directly, the same point made in
  [`NISAR_GCOV_Time_Series_Example.ipynb`](../notebooks/NISAR_GCOV_Time_Series_Example.ipynb)'s
  section on converting between dB and power scale before computing statistics —
  while `..._std3x3` is averaged directly (a simple approximation; a statistically
  rigorous pooled-variance combination wasn't judged worth the extra complexity for
  this rare edge case).
- **Filtering out short-lived tracks.** A relative orbit/track covering a station only
  once or twice in the search window gives too short a time series to be useful. This
  is now filtered out at the query stage — before any granule is opened — by parsing
  the track number and acquisition time directly out of each granule's `native-id`.
:::

:::{note} Design decisions carried over from the previous version
- **Frequency A only** (frequency B, a coarser ~80 m grid, has no column in the
  requested schema) and **L-band only** (this product line has no `/science/SSAR`
  data, so `NISAR_S_HH_db`/`NISAR_S_HV_db` are omitted entirely).
:::

<hr>

## Overview

1. Load station locations
2. Compute local terrain slope/aspect per station from Copernicus GLO-30 (helper)
3. Deduplicate reprocessed product versions (helper)
4. Filter to tracks with enough repeat acquisitions (helper)
5. Merge acquisitions from the same physical overpass (helper)
6. Process each station independently: search, extract, save
7. (Optional) Visualize combined granule coverage across all stations

<hr>

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timezone

from ipyleaflet import CircleMarker
import earthaccess

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from util.aoi_coverage import plot_aoi_coverage
from util.nisar_station_timeseries import (
    compute_station_terrain,
    process_station,
    get_fsspec_https_session_with_timeout,
)

## 1. Load station locations

Build a simple list of station name/lon/lat/elevation from the GeoJSON — no `geopandas` needed here. `elevation_m` (ISMN metadata) is kept for reference only; the height input actually used later to interpolate GCOV's radarGrid cubes is `dem_elevation_m`, read directly from the Copernicus DEM in Section 2 (see the note there for why).

In [2]:
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = [
    {
        "name": feat["properties"]["station_val"],
        "lon": feat["geometry"]["coordinates"][0],
        "lat": feat["geometry"]["coordinates"][1],
        "elevation_m": float(feat["properties"]["elevation_val"]),
    }
    for feat in stations_geojson["features"]
]

TEMPORAL = ("2026-06-15", datetime.now(timezone.utc).strftime("%Y-%m-%d"))

print(f"Loaded {len(stations)} stations")
stations[:3]

Loaded 16 stations


[{'name': 'AtigunPass',
  'lon': -149.47817,
  'lat': 68.12983,
  'elevation_m': 1457.0},
 {'name': 'Coldfoot', 'lon': -150.183, 'lat': 67.25333, 'elevation_m': 326.0},
 {'name': 'EagleSummit',
  'lon': -145.41212,
  'lat': 65.48588,
  'elevation_m': 1106.0}]

## 2. Compute local terrain slope/aspect per station from Copernicus GLO-30

Implemented by `compute_station_terrain()` in
[`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py), shared
across this cookbook's AOI notebooks.

The true **local incidence angle** is the angle between the radar's line-of-sight (LOS)
and the *actual* local surface normal — a vector that tilts with real terrain slope and
aspect, unlike GCOV's own `incidenceAngle`/`elevationAngle`, which are both measured
against the smooth ellipsoid normal and know nothing about terrain tilt. Computing it
needs two independent ingredients per acquisition: the LOS vector's direction (from
GCOV, per-granule — handled in Section 6) and the local surface normal (from a DEM,
computed **once per station here**, since terrain slope/aspect don't change between
acquisitions).

**Source**: [Copernicus DEM GLO-30](https://registry.opendata.aws/copernicus-dem/),
public AWS Open Data, no credentials needed. Tiles are `1°x1°` Cloud-Optimized GeoTIFFs
named by their SW-corner integer degree (e.g. `Copernicus_DSM_COG_10_N41_00_E001_00_DEM`);
reading them via GDAL's `/vsicurl/` driver lets `rasterio` pull just a small windowed
read (a few KB, thanks to COG's internal tiling) instead of downloading the ~25 MB tile.
This AOI's 19 stations, spread across Catalonia/the Pyrenees, likely span a handful of
DEM tiles — tile handles are cached and reused across stations that happen to share
one.

**Method**: a 3x3 pixel window (Horn's method, the standard slope/aspect algorithm used
by ArcGIS/QGIS/GDAL) centered on each station's DEM pixel gives the elevation gradient
in the east/north directions; converting the degree-based pixel spacing to meters
(longitude spacing shrinks with `cos(latitude)`) keeps the gradient in real units. From
slope `β` and aspect `α` (downhill compass bearing, clockwise from north), the unit
surface normal in local East-North-Up coordinates is:

```
normal = (sin(β)·sin(α), sin(β)·cos(α), cos(β))
```

(sanity-checked against the limiting case of a vertical cliff: a south-facing cliff's
normal comes out exactly horizontal and pointing due south, as it should — this same
formula was also cross-checked, for the USA-low-biomass AOI's steepest stations,
against an independent `gdaldem aspect` run on the same DEM tiles, agreeing to within
~2°.)

**Elevation**: the same 3x3 window's center pixel is also read out as `dem_elevation_m`
— free, since the read already happened for the gradient above — and used in Section 6
as the height input for interpolating GCOV's `heightAboveEllipsoid`-indexed radarGrid
cubes, in place of the ISMN-reported `station["elevation_m"]`. This is preferred over
the ISMN value for two reasons: it's read from the exact same pixel driving the
slope/aspect calculation above (self-consistent), and it's more robust than ISMN
station metadata, which varies in quality across networks (missing, stale, or
coarsely-rounded elevations aren't unusual). Neither source is exactly in GCOV's WGS84
ellipsoidal-height datum — GLO-30 is EGM2008-geoid-referenced — but that offset (tens of
meters at most) is negligible here: the radarGrid height levels are spaced far more
coarsely than that, and `incidenceAngle`/the LOS vector both vary very slowly with
height, so a datum offset this small has no meaningful effect on the interpolated
result.

In [3]:
compute_station_terrain(stations)

Computing local terrain slope/aspect from Copernicus GLO-30 for each station...
Done. 6 unique DEM tiles used for 20 stations.
Slope range across stations: 1.00 - 26.93 deg (median 7.14 deg)


## 3. Deduplicate reprocessed product versions

Implemented by `keep_latest_versions()` in
[`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py), used
internally by `process_station()` (Section 6) — not called directly here.

NISAR GCOV granules are occasionally reprocessed with a new
{abbr}`CRID (Composite Release Identifier)`, the same acquisition then appearing twice
in search results under two different CRIDs — the same pattern
[`NISAR_GCOV_Time_Series_Example.ipynb`](../notebooks/NISAR_GCOV_Time_Series_Example.ipynb)
handles by parsing the CRID out of each product's URL and keeping only the newest.
Here we parse it from `native-id` instead (no URL needed), and keep only the
highest-CRID granule per unique acquisition.

## 4. Filter to tracks with enough repeat acquisitions

Implemented by `filter_tracks_with_min_acquisitions()` in
[`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py), used
internally by `process_station()` (Section 6) — not called directly here.

Only keep acquisitions from a relative orbit/track if that track covers the station at
least `MIN_ACQUISITIONS_PER_TRACK` times in the search window — a track seen only once
or twice gives too short a time series to be useful. This is done **at the query
stage**, before any granule is opened: NISAR's granule ID/`native-id` already encodes
both the track number and the acquisition start time
(`NISAR_L2_PR_GCOV_<cycle>_<track>_<A|D>_<frame>_..._<startTime>_<endTime>_<CRID>_...`),
verified against the real `trackNumber` HDF5 attribute (exact match on all granules
checked) — so tracks can be counted and filtered from search-result metadata alone,
before streaming any data. Acquisitions from the same overpass split across adjacent
frames (seconds apart, same track — see the next step) count as a single acquisition
here, using the same time-gap grouping logic.

## 5. Merge acquisitions from the same physical overpass

Implemented by `merge_adjacent_acquisitions()` in
[`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py), used
internally by `process_station()` (Section 6) — not called directly here.

A station near a frame boundary can appear in two adjacent granules from the exact
same pass — acquired only seconds apart, sharing the same `NISAR_orb_number` — rather
than two independent overpasses. These need to be collapsed into a single row before
saving: `NISAR_L_HH_db_mean3x3`/`NISAR_L_HV_db_mean3x3` are averaged **in linear
power** (converting dB → power, averaging, then power → dB), since dB values cannot be
arithmetically averaged directly; `NISAR_L_HH_db_std3x3`/`NISAR_L_HV_db_std3x3` are
averaged directly (a simple approximation for this rare edge case, not a statistically
rigorous pooled-variance combination).

Rows are grouped by temporal proximity: after sorting by time, a new group starts
whenever the gap to the previous acquisition exceeds `ADJACENT_PASS_GAP_SECONDS`
(default 60 s, defined in the previous step). Frames from the same pass land within
tens of seconds of each other; distinct overpasses are separated by hours to days, so
this cleanly tells them apart without needing to also check `orb_number` (though
within a group it is always identical by construction — this is only ever a merge
across a single physical pass).

## 6. Process each station independently: search, extract, save

Implemented by `process_station()` in
[`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py) (which
internally calls `extract_granule_record()` per granule, and the Section 3-5
helpers above). For each station, in turn:

1. **Search** `NISAR_L2_GCOV_PROVISIONAL_V1` using `point=(lon, lat)` — CMR itself
   restricts results to granules whose footprint contains the station, so no separate
   polygon-matching step is needed.
2. **Deduplicate** reprocessed versions with `keep_latest_versions`.
3. **Filter** to tracks with at least `MIN_ACQUISITIONS_PER_TRACK` acquisitions with
   `filter_tracks_with_min_acquisitions`.
4. **Skip granules already saved.** If `{station_name}_NISAR_raw.csv` already exists,
   each candidate granule's track + start time (parsed from `native-id`, no download
   needed) is checked against the existing rows; if every granule is already covered,
   the station is skipped entirely with no network calls beyond the CMR search. This
   makes it cheap to re-run the notebook periodically as `TEMPORAL`'s end date moves
   forward — only newly acquired granules are fetched.
5. For each remaining (new) granule, **open it remotely** with `fsspec` + `h5py` (no
   download) and:
   - Locate the nearest frequency A pixel to the station, then a 3x3 window of
     pixels centered on it (clipped to whatever part falls inside the frequency A
     grid extent, for a station near an edge).
   - Within that window, exclude individual pixels whose `mask` value is `0`
     (partially focused/invalid) or `255` (outside the acquisition extent), and any
     pixel with non-positive `HHHH`/`HVHV` power (`NaN` for the whole window if this
     acquisition isn't HH/HV-polarized, e.g. a VV/VH-mode granule).
   - Convert the remaining pixels' raw `HHHH`/`HVHV` to dB via the product's own
     documented formula, `10*log10(<GCOV_TERM>)`, then compute
     `NISAR_L_HH_db_mean3x3`/`NISAR_L_HV_db_mean3x3` by averaging the window's
     *linear power* values and converting the average to dB (dB values can't be
     arithmetically averaged directly), and `NISAR_L_HH_db_std3x3`/
     `NISAR_L_HV_db_std3x3` as the sample standard deviation (ddof=1) of the
     per-pixel dB values directly (`NaN` if fewer than 2 valid pixels remain in the
     window).
   - Look up the ellipsoidal `incidenceAngle` (used only to reconstruct the LOS
     vector's vertical component, not written out) and the LOS vector's horizontal
     components (`losUnitVectorX`/`losUnitVectorY`) from GCOV's `radarGrid` cubes at
     the station's projected `(y, x)` and `dem_elevation_m` (see the note below for
     why this reads only a couple of scalars per field rather than loading the whole
     cube). Combine the (renormalized) LOS unit vector with the station's precomputed
     DEM surface normal (Section 2) via a dot product to get the true **local
     incidence angle**, recorded as `NISAR_local_incidence_angle_deg` — a
     per-acquisition feature for downstream modeling.
6. **Merge** same-pass adjacent frames with `merge_adjacent_acquisitions`.
7. **Save** the new rows to `{station_name}_NISAR_raw.csv`, appending if the file
   already exists (writing the header only when creating it fresh), before moving on
   to the next station.

:::{important} Why the radarGrid lookups avoid loading the full cubes
`incidenceAngle`, `losUnitVectorX`, and `losUnitVectorY` are each shaped
`(height, y, x)` and chunked `(1, 512, 512)` — around **43 MB per granule, per
field**. An earlier version of this notebook called `RegularGridInterpolator` on the
entire `incidenceAngle` array, so every station x granule combination re-downloaded
the whole 43 MB cube just to read one point; across even a handful of granules per
station this made the notebook take several minutes to stall. `extract_granule_record`
instead reads only the nearest `(y, x)` grid cell and the (at most two) height levels
needed for linear interpolation — a few scalar reads (~2 MB of chunks) instead of the
full cube, verified to match the full interpolation to within ~0.01°. All three fields
share the same `(y, x)` cell and the same two height levels, so the helper
(`_interp_radar_grid_field`, in `util/nisar_station_timeseries.py`) computes the
nearest cell and height bracket once and reuses them for all three — reading three
fields costs three pairs of scalar reads, not three separate full lookups.
:::

In [ ]:
earthaccess.login()

# Was earthaccess.get_fsspec_https_session() -- that builds its session with no
# timeout at all, so a stalled fetch (a connection accepted but that stops sending
# data mid-stream) hangs indefinitely instead of failing and retrying. This version
# adds an explicit timeout; see get_fsspec_https_session_with_timeout's docstring in
# util/nisar_station_timeseries.py for the full story (observed a 17+ minute hang on
# a single HVHV window read before this fix).
#
# Only used directly here for the diagnostic read below -- process_station() now
# opens its own fresh session per granule instead of sharing this one across the
# whole batch (see get_fsspec_https_session_with_timeout's skip_instance_cache
# note: two granule reads sharing one session, one of them a timed-out extraction
# still running in the background, is what caused saved rows' datetime_utc/
# NISAR_orb_dir/NISAR_orb_number to end up naming a different granule than the one
# actually read for HH/HV -- diagnosed via the equivalent bug in the GSLC window
# pipeline, see util/nisar_gslc_windows.py).
fs = get_fsspec_https_session_with_timeout()
fsspec_config = {
    "cache_type": "background",
    "block_size": 8 * 1024 * 1024,  # 8 MB
}

# This notebook was originally written for the Spain AOI; the actual target has
# since moved to Alaska (see Section 1's stations_path) but this output directory
# name was never updated to match -- fixed here so results don't land mislabeled
# under nisar_data/spain/.
output_dir = Path.cwd() / "nisar_data" / "alaska"
output_dir.mkdir(parents=True, exist_ok=True)

all_gpolygons = {}  # native-id -> GPolygon, collected for the optional overview map below

for station in stations:
    process_station(station, output_dir, TEMPORAL, fsspec_config, all_gpolygons)

In [ ]:
import time

# DIAGNOSTIC (temporary): isolate whether the earlier hang was (a) a stalled
# connection the configured timeout should have caught, or (b) the event loop
# thread itself stalled and never got to enforce that timeout, or (c) h5py
# genuinely needs many small, individually-fast reads to open this file's
# structure. This bypasses h5py and the background block-cache entirely --
# one raw small read through the SAME `fs` session confirmed above to have
# timeout=60s configured, so it should resolve one way or another well under
# a minute no matter what.
_diag_results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_PROVISIONAL_V1",
    point=(stations[0]["lon"], stations[0]["lat"]),
    temporal=TEMPORAL,
)
_diag_url = _diag_results[0].data_links(access="external")[0]
print("Testing URL:", _diag_url)

_t0 = time.monotonic()
try:
    with fs.open(_diag_url, cache_type="none") as f:
        chunk = f.read(4096)
    print(f"Raw 4KB read succeeded in {time.monotonic() - _t0:.1f}s, got {len(chunk)} bytes")
except Exception as e:
    print(f"Raw read failed after {time.monotonic() - _t0:.1f}s: {type(e).__name__}: {e}")

## 7. (Optional) Visualize combined granule coverage across all stations

A single overview map built from every unique granule footprint encountered across all per-station searches above, reusing `plot_aoi_coverage` from `util/aoi_coverage.py`.

In [5]:
lons = [s["lon"] for s in stations]
lats = [s["lat"] for s in stations]
pad = 0.05

aoi_wkt = (
    "POLYGON(("
    f"{min(lons) - pad} {min(lats) - pad}, {max(lons) + pad} {min(lats) - pad}, "
    f"{max(lons) + pad} {max(lats) + pad}, {min(lons) - pad} {max(lats) + pad}, "
    f"{min(lons) - pad} {min(lats) - pad}"
    "))"
)

m = plot_aoi_coverage(list(all_gpolygons.values()), aoi_wkt, "GCOV")

for s in stations:
    m.add(CircleMarker(
        location=(s["lat"], s["lon"]),
        radius=4,
        color="red",
        fill_color="red",
        fill_opacity=0.9,
    ))

m

Map(center=[41.570265, 1.1064349999999998], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

<hr>

## Summary

You now have one CSV per station in `nisar_data/spain/`, each named
`{station_name}_NISAR_raw.csv`, with columns `datetime_utc`,
`NISAR_L_HH_db_mean3x3`, `NISAR_L_HH_db_std3x3`, `NISAR_L_HV_db_mean3x3`,
`NISAR_L_HV_db_std3x3`, `NISAR_local_incidence_angle_deg`, `NISAR_orb_dir`,
`NISAR_orb_number` — one row per GCOV acquisition (or same-pass frame group)
covering that station, reprocessed product versions already deduplicated, and tracks
with fewer than `MIN_ACQUISITIONS_PER_TRACK` acquisitions already excluded.
`NISAR_L_HH_db_mean3x3`/`NISAR_L_HV_db_mean3x3` are the 3x3-window mean backscatter in
dB (averaged in linear power, then converted); `NISAR_L_HH_db_std3x3`/
`NISAR_L_HV_db_std3x3` are the sample standard deviation (ddof=1) of the same window's
per-pixel dB values, a simple local texture/spread descriptor.
`NISAR_local_incidence_angle_deg` is the true DEM-corrected local incidence angle
(Section 2), not the ellipsoidal one, and is intended to be usable as a model feature
in its own right (e.g. alongside a separate model per `NISAR_orb_dir`).

These can be joined against the ISMN soil moisture time series produced in
[`01_download_ismn_data.ipynb`](01_download_ismn_data.ipynb) by aligning
`datetime_utc` with the ISMN timestamps.

<hr>